# `xdggs-dggal`: A xdggs plugin using dggal

`xdggs-dggal` is a plugin of xdggs using [dggal](https://dggal.org/) to provide DGGRS backend service. 

The plugin aims to provide different types of DGGS indexing for xarray datasets, the implementation supports the following DGGS:

- IVEA7H (register as ivea7h.dggal in xdggs)
- rHEALPix (register as rhealpix.dggal in xdggs)
- HEALPix (register as healpix.dggal in xdggs)
- ISEA7H_Z7 (register as isea7hz7.dggal in xdggs)


It consists of 2 main functions:

1. Easy access to DGGS indexed xarray datasets, including query the datasets with points in WGS84, or zone ID.
2. Convert raster dataset in GeoTiff format to specific DGGS indexed xarray datasets

## Converting a GeoTiff into a DGGS indexed xarray dataset using the nearest centroid method

### Install the package from `pypi` or install it from GitHub 

In [1]:
#Install it from pypi : 
# !pip install xdggs-dggal
#Or install it from GitHub for the lastest commits.
# pip install git+https://github.com/LandscapeGeoinformatics/xdggs-dggal.git

#Install additional library to load a geotiff into a xarray dataset
# !pip install rioxarray

In [2]:
import xarray as xr
import rioxarray as rio
import os
import warnings
warnings.filterwarnings("ignore")

### Using rioxarray to load the GeoTiff into xarray dataset

In this demonstration, we use a sample collection prepared by the [Landscape and Geoinformatics Lab](https://landscape-geoinformatics.ut.ee/) of the University of Tartu, which is available on Zenodo.

The collection consists of three GeoTiff files (DEM, TWI, Slope). [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18877265.svg)](https://doi.org/10.5281/zenodo.18877265)


In [3]:
# Using DEM sample to demonstrate the conversion, you can also try with others sample using the following links
# Slope: https://zenodo.org/records/18877265/files/est_topo_slope_10m_clipped.tif?download=1
# TWI: https://zenodo.org/records/18877265/files/est_topo_twi_10m_clipped.tif?download=1

tiff_path = "https://zenodo.org/records/18877265/files/est_topo_dem_10m_clipped.tif?download=1"
raster_ds = rio.open_rasterio(tiff_path, band_as_variable=True, masked=True)
raster_ds

<xarray.Dataset> Size: 8MB
Dimensions:      (x: 1779, y: 1081)
Coordinates:
  * x            (x) float64 14kB 6.334e+05 6.335e+05 ... 6.512e+05 6.512e+05
  * y            (y) float64 9kB 6.468e+06 6.468e+06 ... 6.457e+06 6.457e+06
    spatial_ref  int64 8B 0
Data variables:
    band_1       (y, x) float32 8MB ...
Attributes:
    AREA_OR_POINT:  Area

### Convert the xarray dataset into DGGS indexed xarray dataset
In the example notebook, we use DGGRS `isea7h_z7` to demonstrate converting a raster xarray dataset into a DGGS-indexed xarray dataset with the nearest centroid method.

In [4]:
%%time
#To perform conversion using nearest neighbourhood mehtod
from xdggs_dggal.regridding.methods.centroid_based import nearestcentroid
from xdggs_dggal.regridding import regridding
isea7hz7_indexed_ds = regridding(ds=raster_ds,
                               grid_name='isea7hz7.dggal', 
                               method='nearestcentroid',
                               refinement_level=11)

Registered regridding method nearestcentroid
CPU times: user 11.3 s, sys: 442 ms, total: 11.8 s
Wall time: 15.2 s


In [20]:
# Apply mean aggregation to the dataset by zone ID
isea7hz7_indexed_ds = isea7hz7_indexed_ds.groupby('zone_id').mean()
isea7hz7_indexed_ds

<xarray.Dataset> Size: 858kB
Dimensions:      (zone_id: 71512)
Coordinates:
  * zone_id      (zone_id) int64 572kB 6341132698773504716 ... 63411332163170...
    spatial_ref  int64 8B 0
Data variables:
    band_1       (zone_id) float32 286kB 110.1 109.0 109.4 ... 39.41 39.97 39.01
Attributes:
    AREA_OR_POINT:  Area

## Accessing DGGS indexed xarray datasets using `xdggs-dggal`

After the conversion is done, the dataset is indexed with DGGS. However, the index is not meaningful and cannot be used by others. Therefore, `xdggs-dggal` provides the `DGGALIndex` as an interface between the user and the zone ID. It is implemented as a plugin of [xdggs](https://github.com/xarray-contrib/xdggs/tree/main).

In [21]:
# import xdggs and register IGEO7Index by importing the IGEO7Index
import xdggs
from xdggs_dggal.index import DGGALIndex

In [22]:
# since the xdggs convention uses the name `cell_ids` as the DGGS zone ID, we have to rename the `zone_id` to `cell_ids`
isea7hz7_indexed_ds = isea7hz7_indexed_ds.rename({'zone_id': 'cell_ids'})

### Create a DGGALIndex instance for the DGGS dataset
- using the xdggs.decode function to create a DGGALIndex object for the dataset
- From the output, the `decode` function created a DGGALIndex object using the attributes from `cell_ids` (zone_id)

In [23]:
isea7hz7_indexed_ds = isea7hz7_indexed_ds.pipe(xdggs.decode)
isea7hz7_indexed_ds

<xarray.Dataset> Size: 858kB
Dimensions:      (cell_ids: 71512)
Coordinates:
  * cell_ids     (cell_ids) int64 572kB 6341132698773504716 ... 6341133216317...
    spatial_ref  int64 8B 0
Data variables:
    band_1       (cell_ids) float32 286kB 110.1 109.0 109.4 ... 39.97 39.01
Indexes:
    cell_ids  DGGALIndex(grid=RHEALPIX.DGGAL,level=11)
Attributes:
    AREA_OR_POINT:  Area

### Using DGGALIndex to access the dataset
- xdggs provides several accessor functions through `<dataset>.dggs`
  - `sel_latlon`:
  - `cell_boundaries`:
  - `explore`:

In [24]:
lats = [58.28459946, 58.27637829, 58.26980135]
lons = [26.41758320, 26.42799669,  26.36551576]
isea7h_indexed_subset = isea7hz7_indexed_ds.dggs.sel_latlon(lats, lons)
isea7h_indexed_subset

<xarray.Dataset> Size: 44B
Dimensions:      (cell_ids: 3)
Coordinates:
  * cell_ids     (cell_ids) int64 24B 6341132975798895202 ... 634113303592843...
    spatial_ref  int64 8B 0
Data variables:
    band_1       (cell_ids) float32 12B 46.71 53.47 83.06
Indexes:
    cell_ids  DGGALIndex(grid=RHEALPIX.DGGAL,level=11)
Attributes:
    AREA_OR_POINT:  Area

In [25]:
isea7hz7_indexed_ds['band_1'].dggs.explore()